# Overview: the five stages

`lin_ode` rewrites a nonlinear ODE as a linear one. A matrix exponential then solves it.

Run these notebooks with `uv run --group docs jupyter lab notebooks`.

## The problem

The state is a vector over $N$ spatial sites, $u=(u_0,\dots,u_{N-1})$. It obeys $\dot u=f(u)$, where
$f$ is a polynomial. The equation is nonlinear, so it has no matrix exponential.

Two ideas together remove the nonlinearity:

| idea | what it does |
| --- | --- |
| **Carleman** | adds monomials of $u$ as extra coordinates: $\Phi=(1,\ u_s,\ u_su_t,\ \dots)$ up to total degree $K$ |
| **Koopman** | writes the exact *linear* equation those coordinates obey, $\dot{\Phi}=A\Phi$ |

Each entry of $\Phi$ is a **basis function**. $1$, $u_2$ and $u_0u_3^{2}$ are three of them. The
**basis** is the whole list, and $K$ is the largest total degree it keeps.

Truncating at degree $K$ makes $A$ finite. It is also very large:

$$\dim A=\binom{N+K}{K}
\qquad\longrightarrow\qquad
5\times10^{17}\ \text{ at } N=10^{9},\ K=2$$

`lin_ode` does not compute all of $A$ in advance. You ask for one entry, and it computes that entry
alone, in closed form. The solution is still $e^{At}\Phi(0)$, but you never hold $A$.

The entries you ask for belong to the **generator** $L$, the operator Koopman produces. $L$
differentiates along the flow. $A$ is the transpose of its matrix. The two differ because $L$ acts
on observables and $A$ acts on the coordinate vector $\Phi$ ([`05_matrix`](05_matrix.ipynb)).

## The five stages

| # | stage | class | tutorial |
| --- | --- | --- | --- |
| 1 | the ODE | `OrdinaryDifferentialSystem` | [`02_ode`](02_ode.ipynb) |
| 2 | the linearization | `LinearOperator` | [`03_linearization`](03_linearization.ipynb) |
| 3 | the basis | `TruncatedBasisSet` | [`04_basis`](04_basis/index.ipynb) |
| 4 | the matrix | `OperatorMatrix` | [`05_matrix`](05_matrix.ipynb) |
| 5 | the vectors | `Vector` | [`06_vector`](06_vector.ipynb) |

| track | stages | produces |
| --- | --- | --- |
| operator | 1 → 2 → 4 | the matrix |
| vector | 5 | the initial condition and the observables |

Both tracks use the basis from stage 3. Stage 3 therefore comes before stages 4 and 5.

Stage 2 offers three linearizations: `Koopman`, `Liouville` and `KoopmanVonNeumann`. They differ in
which object evolves. The three choices are the observable, the density and the wavefunction. This
page uses `Koopman`.

Stage 3 offers three bases: `CarlemanBasis`, `PSCarlemanBasis` and `GaussianHermiteBasis`. All three
are subclasses of `TruncatedBasisSet`. This page uses `CarlemanBasis`.

In the diagram, blue is a `lin_ode` class. Amber is a choice you make. Grey is your input. Each
arrow is the call that produces the next box, and its label is the method name.

![The five stages: your equation into an operator, a basis, and a matrix or vector.](../figures/five_stages.png)


`OperatorMatrix` and `Vector` both work this way. Building them computes nothing. Each one computes
an element when you ask for that element, and the cost is per element.

## Labels and entries

A **label** names a basis function.

| label | basis function |
| --- | --- |
| `()` | $1$ |
| `(2, 1)` | $u_2$ |
| `(2, 2)` | $u_2^{2}$ |
| `((0, 1), (3, 2))` | $u_0\,u_3^{2}$ |

You may write a single factor flat: `(2, 1)` and `((2, 1),)` are the same label. Nest two or more.

An entry of the matrix sits where one basis function meets another, so `matrix[bra, ket]` takes two
labels.

- `bra` picks the row, and `ket` picks the column.
- The entry is the coefficient of `bra` in $L\lvert\text{ket}\rangle$. The two names come from the
  notation for it, $\langle\text{bra}\rvert L\lvert\text{ket}\rangle$.

## All five stages in one cell

KPP–Fisher (Kolmogorov–Petrovsky–Piskunov) is the standard reaction–diffusion test equation. On four
sites $i=0,1,2,3$:

$$\frac{du_i}{dt}=\frac{D\,(u_{i-1}-2u_i+u_{i+1})}{h^{2}}+Q\,u_i+R\,u_i^{2}$$

| symbol | meaning |
| --- | --- |
| $D$ | diffusion coefficient |
| $Q$ | linear growth rate |
| $R$ | nonlinear reaction strength |
| $h$ | grid spacing |

An index that runs off the grid **wraps**. `lin_ode` reads $u_{-1}$ as $u_3$ and $u_4$ as $u_0$, so
these four sites form a ring and every site has two neighbours.

An open end is not the default, so you have to specify one. Either pass the edge rule through
`conditions`, or write the branch into the equation yourself, so that a neighbour outside the grid
contributes 0.

[`02_ode`](02_ode.ipynb) states the wrapping rule in general, and
[`07_boundary_conditions`](07_boundary_conditions.ipynb) covers `conditions`.

In [1]:
import sympy as sp

from lin_ode.basis import CarlemanBasis
from lin_ode.operators import Koopman
from lin_ode import OperatorMatrix, OrdinaryDifferentialSystem
from lin_ode.vector import Vector

D, Q, R, h = sp.symbols("D Q R h")
u, i = sp.IndexedBase("u"), sp.Idx("i", (0, 3))

ode = OrdinaryDifferentialSystem.from_sympy(                              # 1. the ODE
    D * (u[i - 1] - 2 * u[i] + u[i + 1]) / h**2 + Q * u[i] + R * u[i] ** 2, u, (i,)
)
operator = Koopman().create_operator(ode)                                 # 2. the linearization
basis = CarlemanBasis(n_vars=4, degree=2)                                 # 3. the basis
matrix = OperatorMatrix(basis, operator)                                  # 4. the matrix
observable = Vector(u[0] ** 2 + u[1] ** 2, basis, u)                      # 5. an observable

print("basis size            ", len(basis))
print("< u_2^2 | L | u_2 >   ", matrix[(2, 2), (2, 1)])
print("< u_1   | L | u_2 >   ", matrix[(1, 1), (2, 1)])
print("coefficient of u_0^2  ", observable[(0, 2)])

basis size             15
< u_2^2 | L | u_2 >    R
< u_1   | L | u_2 >    D/h**2
coefficient of u_0^2   1


An **observable** is a function of the state you measure, such as an energy. Its **expectation
value** is one number. That number summarises the state at a given time, and it is the quantity you
plot.

The four printed lines:

- `basis size 15` is the $\binom{N+K}{K}$ from the top of this page, at $N=4$ and $K=2$.
- $\langle u_2^{2}\rvert L\lvert u_2\rangle=R$, because the reaction term $R\,u_i^{2}$ couples $u_2$
  to $u_2^{2}$.
- $\langle u_1\rvert L\lvert u_2\rangle=D/h^{2}$, the diffusion term reaching the neighbour.
- A `Vector` is an object you index into to get the observable's component on one basis function.
  Here the observable is $u_0^{2}+u_1^{2}$ and `(0, 2)` names $u_0^{2}$, so `observable[(0, 2)]` is
  that monomial's coefficient, $1$.

Answers are symbolic. Substitute numbers when you need them.

`Vector(expr, basis, u)` takes `u` as a separate argument, so it knows which `IndexedBase` in `expr`
is the state.

`n_vars=4` is the only place the grid size appears. The same labels on a $10^{9}$-site basis return
the same closed forms at the same cost ([`05_matrix`](05_matrix.ipynb)).

---

[Notebook index](../README.md) · [1. The ODE →](02_ode.ipynb)